## Tahap 0: Menentukan Lokasi Folder Dataset

Sel ini mencari root project secara otomatis dengan cara naik dari folder saat ini
sampai ketemu folder yang punya subfolder `Praktikum`.

Tujuannya supaya notebook tetap jalan baik dibuka dari PyCharm maupun Jupyter.
Kalau pathnya ditulis manual, notebook cuma jalan di satu komputer dan langsung
error begitu pindah folder.

In [1]:
from pathlib import Path


def cari_root(mulai: Path) -> Path:
    """Cari root project, yaitu folder yang memuat subfolder 'Praktikum'."""
    for p in (mulai, *mulai.parents):
        if (p / "Praktikum").is_dir():
            return p
    return mulai


ROOT = cari_root(Path.cwd().resolve())
DATASET_DIR = ROOT / "Dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

FILE_MENTAH = DATASET_DIR / "dataset_leetcode_raw.csv"      # cache hasil scraping
FILE_XLSX = DATASET_DIR / "dataset_leetcode_cleaned.xlsx"   # keluaran utama

print(f"Root project  : {ROOT}")
print(f"Folder dataset: {DATASET_DIR}")

Root project  : /Users/alif/PycharmProjects/Text-Mining
Folder dataset: /Users/alif/PycharmProjects/Text-Mining/Dataset


## Tahap 1: Import Library

Empat library untuk tahap pengambilan data:

`requests` mengirim permintaan ke API LeetCode, `pandas` menampung hasilnya dalam
bentuk tabel, `BeautifulSoup` membaca HTML deskripsi soal, dan `time` memberi jeda
antar permintaan supaya tidak diblokir server.

In [2]:
# ==========================================
# TAHAP 1: IMPORT LIBRARY
# ==========================================
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time # Untuk memberi jeda agar tidak diblokir LeetCode

### Library untuk pemrosesan teks

Sel berikut menyiapkan NLTK, library yang dipakai untuk tokenizing, stopword, dan
lemmatization.

Ada tambahan blok perbaikan sertifikat SSL di atas perintah download. Di macOS,
`nltk.download()` sering gagal dengan pesan `CERTIFICATE_VERIFY_FAILED` karena
Python tidak menemukan sertifikat root sistem. Blok itu mengarahkan Python ke
sertifikat milik paket `certifi`, jadi unduhan bisa jalan.

Empat resource yang diunduh: `punkt` dan `punkt_tab` untuk memecah kalimat jadi
kata, `stopwords` untuk daftar kata umum bahasa Inggris, dan `wordnet` untuk
kamus lemmatization.

In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Perbaikan sertifikat SSL, khusus macOS.
# Tanpa blok ini, nltk.download() gagal dengan CERTIFICATE_VERIFY_FAILED
# karena Python tidak menemukan sertifikat root sistem.
import ssl

try:
    import certifi

    ssl._create_default_https_context = (
        lambda *a, **kw: ssl.create_default_context(cafile=certifi.where())
    )
except ImportError:
    ssl._create_default_https_context = ssl._create_unverified_context

# Unduh resource (hanya perlu dijalankan sekali)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/alif/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/alif/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/alif/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/alif/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Tahap 2: Menyiapkan Query GraphQL

LeetCode tidak menyediakan file dataset, tapi situsnya memakai API GraphQL yang
bisa diakses publik. Jadi datanya diambil lewat API itu, bukan dengan membaca
halaman HTML satu per satu.

Perlu dua query karena API memisahkan datanya:

`query_problem_list` mengambil daftar soal beserta judul, slug, difficulty, dan
tag topiknya. Query ini tidak mengirim isi deskripsi soal.

`query_problem_content` mengambil isi deskripsi satu soal berdasarkan slug yang
didapat dari query pertama.

Konsekuensinya, untuk 90 soal dibutuhkan 90 kali permintaan deskripsi, ditambah
beberapa permintaan daftar.

In [4]:
# Endpoint GraphQL LeetCode
url = "https://leetcode.com/graphql"

# ==========================================
# TAHAP 2: DEFINISI QUERY GRAPHQL
# ==========================================
# Query pertama: Untuk mengambil daftar soal, difficulty, dan tag
query_problem_list = """
query problemsetQuestionList($limit: Int, $skip: Int) {
  problemsetQuestionList: questionList(
    categorySlug: ""
    limit: $limit
    skip: $skip
    filters: {}
  ) {
    questions: data {
      title
      titleSlug
      difficulty
      topicTags {
        name
      }
    }
  }
}
"""

# Query kedua: Untuk mengambil isi deskripsi berdasarkan "titleSlug"
query_problem_content = """
query questionData($titleSlug: String!) {
  question(titleSlug: $titleSlug) {
    content
  }
}
"""

## Tahap 3: Fungsi Scraping

Ini bagian inti pengambilan data. Ada dua fungsi.

**`kirim_query`** membungkus permintaan HTTP dan mengulanginya sampai tiga kali
kalau gagal. Proses scraping penuh memakan waktu sekitar sepuluh menit, jadi satu
gangguan jaringan di tengah jalan tanpa retry akan membatalkan semuanya.

**`get_leetcode_data`** berjalan dengan `while`, bukan `for`, dan ini poin yang
penting. Jumlah soal yang lolos filter tidak bisa ditebak di awal, karena banyak
soal yang dibuang. Jadi loopnya terus mengambil batch berikutnya sampai kuota
benar benar terpenuhi.

Dua filter yang dipakai:

Soal premium dilewati, ditandai dengan `content_html` bernilai `None` karena
isinya memang tidak dikirim server untuk akun gratis.

Soal bergambar dilewati lewat pengecekan `soup.find('img')`. Ini alasannya kenapa
kuota 90 soal butuh menelusuri jauh lebih banyak kandidat, sebab sebagian besar
soal LeetCode memakai gambar untuk ilustrasi.

`time.sleep(1)` di akhir tiap soal adalah jeda supaya server tidak menganggap ini
serangan dan memblokir IP.

In [5]:
def kirim_query(query, variables, max_retry=3):
    """Kirim request GraphQL dengan retry, supaya proses panjang tidak gagal total."""
    for percobaan in range(1, max_retry + 1):
        try:
            response = requests.post(url, json={'query': query, 'variables': variables}, timeout=30)
            response.raise_for_status()
            return response.json()['data']
        except Exception as e:
            if percobaan == max_retry:
                raise
            print(f"   (gagal, mencoba lagi {percobaan}/{max_retry}: {e})")
            time.sleep(2 * percobaan)


def get_leetcode_data(jumlah_soal=90):
    print(f"Mulai mencari tepat {jumlah_soal} soal tanpa gambar...")

    hasil_akhir = []
    skip_soal = 0
    batch_size = 40 # Menarik 40 soal tiap putaran agar efisien

    # Loop akan terus berjalan SELAMA jumlah data valid belum mencapai target
    while len(hasil_akhir) < jumlah_soal:
        variables_list = {"limit": batch_size, "skip": skip_soal}
        data_soal = kirim_query(query_problem_list, variables_list)['problemsetQuestionList']['questions']

        # Hentikan pencarian jika server LeetCode sudah tidak mengembalikan soal (habis)
        if not data_soal:
            print("Pencarian dihentikan. Tidak ada sisa soal di database.")
            break

        for soal in data_soal:
            # Jika kuota jumlah_soal tiba-tiba terpenuhi di tengah proses, langsung hentikan
            if len(hasil_akhir) >= jumlah_soal:
                break

            title = soal['title']
            slug = soal['titleSlug']
            difficulty = soal['difficulty']
            tags = ", ".join([tag['name'] for tag in soal['topicTags']])

            # Request Isi Deskripsi
            variables_content = {"titleSlug": slug}
            content_html = kirim_query(query_problem_content, variables_content)['question']['content']

            # Lewati jika konten premium (kosong)
            if content_html is None:
                continue

            soup = BeautifulSoup(content_html, 'html.parser')

            # Lewati jika ada gambar
            if soup.find('img'):
                continue

            # Jika lolos semua filter, bersihkan dan simpan
            clean_text = soup.get_text(separator=" ", strip=True)
            hasil_akhir.append({
                'Problem Name': title,
                'Category': tags,
                'Difficulty': difficulty,
                'Description': clean_text
            })

            # Indikator progres berjalan
            print(f"Terkumpul {len(hasil_akhir)}/{jumlah_soal} | Berhasil mengambil: {title}")
            time.sleep(1)

        # Maju ke halaman (batch) soal berikutnya untuk dicari
        skip_soal += batch_size

    df = pd.DataFrame(hasil_akhir)
    return df

## Tahap 4: Menjalankan Scraping

Sel ini yang benar benar mengeksekusi pengambilan data.

Ada sakelar `PAKAI_CACHE`. Kalau diisi `True` dan file hasil scraping sebelumnya
masih ada, datanya dibaca dari file itu. Kalau `False`, scraping dijalankan dari
awal ke LeetCode.

Bedanya besar: baca cache selesai dalam hitungan detik, sedangkan scraping penuh
butuh sekitar sepuluh menit. Untuk presentasi, biarkan `True` supaya tidak ada
jeda menunggu.

Hasil scraping mentah langsung disimpan ke CSV begitu selesai, jadi datanya tidak
hilang kalau kernel direstart.

In [6]:
# ==========================================
# TAHAP 3: EKSEKUSI
# ==========================================
JUMLAH_SOAL = 90
PAKAI_CACHE = True   # True = pakai hasil scraping sebelumnya kalau file-nya ada

if PAKAI_CACHE and FILE_MENTAH.exists():
    df_leetcode = pd.read_csv(FILE_MENTAH)
    print(f"Memakai data tersimpan: {FILE_MENTAH.name} ({len(df_leetcode)} soal)")
else:
    df_leetcode = get_leetcode_data(jumlah_soal=JUMLAH_SOAL)
    df_leetcode.to_csv(FILE_MENTAH, index=False)
    print(f"\nData mentah disimpan ke: {FILE_MENTAH}")

Memakai data tersimpan: dataset_leetcode_raw.csv (90 soal)


## Tahap 5: Memeriksa Hasil Scraping

Empat sel berikut memastikan datanya benar sebelum lanjut ke preprocessing.

Sel pertama menampilkan lima baris teratas untuk melihat bentuk datanya: ada empat
kolom, yaitu nama soal, kategori topik, tingkat kesulitan, dan deskripsi.

In [7]:
# Menampilkan 5 data teratas
print("\n--- HASIL DATA FRAME ---")
display(df_leetcode.head())


--- HASIL DATA FRAME ---


,Problem Name,Category,Difficulty,Description
0,Two Sum,"Array, Hash Table",Easy,You are given an array of integers nums and an...
1,Longest Substring Without Repeating Characters,"Hash Table, String, Sliding Window",Medium,"Given a string s , find the length of the long..."
2,Median of Two Sorted Arrays,"Array, Binary Search, Divide and Conquer",Hard,Given two sorted arrays nums1 and nums2 of siz...
3,Longest Palindromic Substring,"Two Pointers, String, Dynamic Programming, Man...",Medium,"Given a string s , return the longest palindro..."
4,Zigzag Conversion,String,Medium,"The string ""PAYPALISHIRING"" is written in a zi..."


Lima baris terakhir dicek juga. Gunanya memastikan proses scraping berjalan utuh
sampai soal ke 90, bukan berhenti di tengah dengan baris kosong.

In [8]:
display(df_leetcode.tail())

,Problem Name,Category,Difficulty,Description
85,Maximum Product Subarray,"Array, Dynamic Programming",Medium,"Given an integer array nums , find a subarray ..."
86,Find Minimum in Rotated Sorted Array,"Array, Binary Search",Medium,Suppose an array of length n sorted in ascendi...
87,Find Minimum in Rotated Sorted Array II,"Array, Binary Search",Hard,Suppose an array of length n sorted in ascendi...
88,Min Stack,"Stack, Design",Medium,"Design a stack that supports push, pop, top, a..."
89,Find Peak Element,"Array, Binary Search",Medium,A peak element is an element that is strictly ...


`info()` menunjukkan tipe data tiap kolom dan berapa nilai yang tidak kosong.

Yang perlu diperhatikan: semua kolom harus menunjukkan 90 non null. Kalau ada yang
kurang, berarti ada soal yang deskripsinya gagal terambil dan perlu ditangani
sebelum preprocessing.

In [9]:
df_leetcode.info()

<class 'pandas.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Problem Name  90 non-null     str  
 1   Category      90 non-null     str  
 2   Difficulty    90 non-null     str  
 3   Description   90 non-null     str  
dtypes: str(4)
memory usage: 2.9 KB


Sel ini menampilkan ukuran dataset dan sebaran tingkat kesulitannya.

Kolom `Difficulty` inilah yang nanti jadi label pada tahap klasifikasi, jadi
sebarannya perlu dilihat sejak awal. Sebarannya tidak seimbang karena soal diambil
berurutan dari nomor satu, dan LeetCode memang punya lebih banyak soal Medium.
Ketidakseimbangan ini perlu ditangani di tahap klasifikasi, misalnya dengan
stratified split.

In [10]:
print("Ukuran dataset:", df_leetcode.shape)
print("\nDistribusi Difficulty (label untuk klasifikasi nanti):")
display(df_leetcode['Difficulty'].value_counts())

Ukuran dataset: (90, 4)

Distribusi Difficulty (label untuk klasifikasi nanti):


Difficulty
Medium    52
Hard      20
Easy      18
Name: count, dtype: int64

## Tahap 6: Preprocessing Teks

Teks mentah tidak bisa langsung dipakai. Kalimat seperti "Given an array nums[i],
return the indices" masih penuh tanda baca, angka, dan kata umum yang tidak
membedakan apa apa.

Fungsi `text_preprocessing` menjalankan lima langkah berurutan:

**Case folding**, semua huruf dijadikan kecil. Tanpa ini, "Array" dan "array"
dihitung sebagai dua kata berbeda.

**Cleansing**, `re.sub(r'[^a-z\s]', '', text)` membuang semua karakter selain
huruf dan spasi. Angka, tanda baca, dan simbol hilang di sini.

**Tokenizing**, kalimat dipecah jadi daftar kata memakai `word_tokenize`.

**Stopword removal**, kata umum seperti "the", "is", "and" dibuang. Ditambah
stopword khusus LeetCode seperti "example", "input", "output", karena kata kata
itu muncul di hampir semua soal sehingga tidak membantu membedakan tingkat
kesulitan.

**Lemmatization**, kata dikembalikan ke bentuk dasarnya. "arrays" jadi "array",
"numbers" jadi "number". Tanpa ini, bentuk jamak dan tunggal dihitung sebagai dua
fitur berbeda dan kosakata jadi membengkak tanpa manfaat.

In [11]:
lemmatizer = WordNetLemmatizer()

# 1. Mengambil stopwords bawaan NLTK (bahasa Inggris)
stop_words = set(stopwords.words('english'))

# 2. Menambahkan Custom Stopwords khusus domain LeetCode
custom_stopwords = [
    'example', 'input', 'output', 'explanation', 'constraint', 'constraints',
    'given', 'return', 'returns', 'true', 'false', 'null'
]
stop_words.update(custom_stopwords)


def text_preprocessing(text):
    # Case Folding & Cleansing (hanya menyisakan huruf a-z)
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenizing
    tokens = word_tokenize(text)

    # Stopword Removal (sekarang sudah termasuk custom_stopwords)
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens]

    return ' '.join(lemmatized)

Fungsi tadi diterapkan ke seluruh 90 baris memakai `apply`, hasilnya disimpan di
kolom baru `Cleaned_Description`.

Kolom asli `Description` sengaja dipertahankan supaya bisa dibandingkan. Kolom
baru inilah yang nanti divektorisasi di notebook kedua.

In [12]:
print("Memulai proses Text Preprocessing pada kolom Description...")

# Menerapkan fungsi ke kolom Description dan menyimpannya di kolom baru
df_leetcode['Cleaned_Description'] = df_leetcode['Description'].apply(text_preprocessing)

print("Proses selesai! Berikut perbandingan hasilnya:\n")

# Menampilkan perbandingan teks sebelum dan sesudah diproses
display(df_leetcode[['Description', 'Cleaned_Description']].head())

Memulai proses Text Preprocessing pada kolom Description...


Proses selesai! Berikut perbandingan hasilnya:



,Description,Cleaned_Description
0,You are given an array of integers nums and an...,array integer nums integer target index two nu...
1,"Given a string s , find the length of the long...",string find length longest substring without d...
2,Given two sorted arrays nums1 and nums2 of siz...,two sorted array nums nums size n respectively...
3,"Given a string s , return the longest palindro...",string longest palindromic substring babad bab...
4,"The string ""PAYPALISHIRING"" is written in a zi...",string paypalishiring written zigzag pattern n...


### Melihat efek preprocessing pada satu contoh

Sel ini mencetak satu soal secara utuh, sebelum dan sesudah diproses. Ini bagian
yang paling jelas untuk ditunjukkan saat presentasi.

Perhatikan apa yang hilang: kalimat "You are given an array of integers nums" jadi
"array integer nums" saja, seluruh blok Example dan Constraints menyusut drastis
karena isinya didominasi angka dan simbol.

Satu catatan jujur soal efek samping: karena angka dibuang, `nums1` dan `nums2`
sama sama jadi `nums`, dan `nums.length` jadi `numslength`. Ini terlihat nanti di
notebook kedua sebagai bigram "nums nums". Bukan error, tapi konsekuensi dari
aturan cleansing yang dipakai.

In [13]:
print("SEBELUM PREPROCESSING:")
print(df_leetcode["Description"][0])
print("\nSESUDAH PREPROCESSING:")
print(df_leetcode["Cleaned_Description"][0])

SEBELUM PREPROCESSING:
You are given an array of integers nums and an integer target , return indices of the two numbers such that they add up to target . You may assume that each input would have exactly one solution , and you may not use the same element twice. You can return the answer in any order. Example 1: Input: nums = [2,7,11,15], target = 9 Output: [0,1] Explanation: Because nums[0] + nums[1] == 9, we return [0, 1]. Example 2: Input: nums = [3,2,4], target = 6 Output: [1,2] Example 3: Input: nums = [3,3], target = 6 Output: [0,1] Constraints: 2 <= nums.length <= 10 4 -10 9 <= nums[i] <= 10 9 -10 9 <= target <= 10 9 Only one valid answer exists. Follow-up: Can you come up with an algorithm that is less than O(n 2 ) time complexity?

SESUDAH PREPROCESSING:
array integer nums integer target index two number add target may assume would exactly one solution may use element twice answer order nums target nums nums nums target nums target numslength numsi target one valid answer exi

## Tahap 7: Menyimpan Dataset

Hasil akhir disimpan ke `Dataset/dataset_leetcode_cleaned.xlsx`. Berkas inilah
yang dibaca oleh `02_Vektorisasi.ipynb`, jadi ini titik sambung antara kedua
notebook.

Sel ini juga membaca ulang file yang baru ditulis sebagai verifikasi. Gunanya
memastikan file benar benar tersimpan dan kolom `Cleaned_Description` ada di
dalamnya, sehingga notebook kedua dijamin tidak gagal saat membacanya.

In [14]:
# Simpan hasil akhir ke format Excel di dalam folder Dataset
df_leetcode.to_excel(FILE_XLSX, index=False)

print(f"Tersimpan: {FILE_XLSX}")
print(f"\nUkuran dataset akhir : {df_leetcode.shape[0]} baris x {df_leetcode.shape[1]} kolom")
print(f"Kolom                : {list(df_leetcode.columns)}")

# Verifikasi: baca ulang berkasnya, persis seperti yang nanti dilakukan notebook 2
cek = pd.read_excel(FILE_XLSX)
print(f"\nVerifikasi baca ulang: {cek.shape[0]} baris, kolom 'Cleaned_Description' "
      f"{'ADA' if 'Cleaned_Description' in cek.columns else 'TIDAK ADA'}")

Tersimpan: /Users/alif/PycharmProjects/Text-Mining/Dataset/dataset_leetcode_cleaned.xlsx

Ukuran dataset akhir : 90 baris x 5 kolom
Kolom                : ['Problem Name', 'Category', 'Difficulty', 'Description', 'Cleaned_Description']

Verifikasi baca ulang: 90 baris, kolom 'Cleaned_Description' ADA


## Selesai

Dataset sudah tersimpan di `Dataset/dataset_leetcode_cleaned.xlsx` dengan 90 baris
dan 5 kolom.

Lanjut ke `02_Vektorisasi.ipynb` untuk mengubah kolom `Cleaned_Description` jadi
angka memakai Bag of Words dan TF-IDF.